# 43 — Integrated Pipeline Specifies Open Model Distribution

**Leading specification:** open model distribution is the composition of identity, verification, chunking, replication, discovery, scheduling, reassembly, and final verification.

Previous notebooks introduced one engineering layer at a time:

| Notebook | Specification |
|---|---|
| 07 | Content addressing specifies identity |
| 13 | Cryptographic verification specifies trust |
| 17 | Chunking specifies transfer units |
| 23 | Replication specifies availability |
| 29 | Discovery specifies retrieval |
| 37 | Scheduling specifies efficient distribution |

Notebook 43 combines them into a single executable pipeline.

## 1. Context

The complete open model distribution workflow is:

```text
content
→ content addressing
→ verification identity
→ chunking
→ replication
→ discovery
→ scheduling
→ download
→ chunk verification
→ reassembly
→ final artifact verification
→ usable model
```

The final acceptance rule is strict:

\[
H(C_{\mathrm{reassembled}})=H(C_{\mathrm{published}})
\]

The model is usable only after the final artifact verifies.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import shutil
import sys
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Robust paths whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "43_integrated_pipeline"
SITE = REPO_ROOT / "site"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
except Exception:
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")

## 2. Integrated variables

Content identity:

\[
I=H(C)
\]

Chunk identities:

\[
I_i=H(c_i)
\]

Discovery over replicas:

\[
D(I_i)\rightarrow\{L_{i1},L_{i2},\ldots\}
\]

Scheduling objective:

\[
s^\*=\arg\min_s \operatorname{cost}(s)
\]

Reassembly:

\[
C=c_0\Vert c_1\Vert\cdots\Vert c_{n-1}
\]

Final verification:

\[
H(C)=I
\]

In [ ]:
@dataclass(frozen=True)
class IntegratedVariables:
    content_identity: str
    chunk_identity: str
    discovery: str
    scheduling: str
    reassembly: str
    final_verification: str

variables = IntegratedVariables(
    content_identity="I=H(C), the published artifact identity",
    chunk_identity="I_i=H(c_i), each transfer unit identity",
    discovery="D(I_i) maps chunk identity to candidate locations",
    scheduling="s* orders candidate transfers by cost",
    reassembly="C is reconstructed from verified chunks in manifest order",
    final_verification="H(C_reassembled)=I accepts the final artifact",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})

## 3. Publish artifact identity

Create a deterministic toy model artifact and publish its digest.

In [ ]:
artifact_dir = RESULTS / "artifact"
chunk_dir = RESULTS / "chunks"
replica_root = RESULTS / "replicas"
download_dir = RESULTS / "downloads"
reassembly_dir = RESULTS / "reassembly"

for d in [artifact_dir, chunk_dir, replica_root, download_dir, reassembly_dir]:
    d.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / "toy-open-model.bin"

payload = (
    b"OPEN MODEL DISTRIBUTION INTEGRATED PIPELINE\\n"
    + b"Notebook 43 composes identity, verification, chunking, replication, discovery, scheduling, and reassembly.\\n"
    + bytes(range(256)) * 64
)

artifact_path.write_bytes(payload)

artifact_digest = sha256_file(artifact_path)
artifact_id = content_id(artifact_path)

artifact_identity = {
    "artifact": str(artifact_path.relative_to(REPO_ROOT)),
    "size_bytes": artifact_path.stat().st_size,
    "algorithm": "sha256",
    "digest": artifact_digest,
    "content_id": artifact_id,
}

artifact_identity_path = RESULTS / "43_artifact_identity.json"
artifact_identity_path.write_text(json.dumps(artifact_identity, indent=2), encoding="utf-8")

artifact_identity

## 4. Chunk the artifact

Split the artifact into fixed-size transfer units and record chunk identities.

In [ ]:
CHUNK_SIZE = 2048
data = artifact_path.read_bytes()

chunk_rows = []
for index, start in enumerate(range(0, len(data), CHUNK_SIZE)):
    chunk = data[start:start + CHUNK_SIZE]
    chunk_path = chunk_dir / f"chunk_{index:04d}.bin"
    chunk_path.write_bytes(chunk)
    digest = sha256_file(chunk_path)
    chunk_rows.append({
        "chunk_index": index,
        "offset_start": start,
        "offset_end": start + len(chunk),
        "size_bytes": len(chunk),
        "path": str(chunk_path.relative_to(REPO_ROOT)),
        "sha256": digest,
        "content_id": f"sha256:{digest}",
    })

chunk_plan = pd.DataFrame(chunk_rows)
chunk_plan_csv = RESULTS / "43_chunk_plan.csv"
chunk_plan.to_csv(chunk_plan_csv, index=False)

chunk_manifest = {
    "schema": "open-model-distribution.integrated-chunk-manifest.v0",
    "artifact_content_id": artifact_id,
    "artifact_digest": artifact_digest,
    "chunk_size_bytes": CHUNK_SIZE,
    "chunk_count": len(chunk_plan),
    "chunks": chunk_plan.to_dict(orient="records"),
}

chunk_manifest_path = RESULTS / "43_chunk_manifest.json"
chunk_manifest_path.write_text(json.dumps(chunk_manifest, indent=2), encoding="utf-8")

chunk_plan

## 5. Replicate chunks

Create candidate sources for each chunk. One candidate is intentionally corrupted, so the integrated pipeline demonstrates fallback and verification.

In [ ]:
sources = pd.DataFrame([
    {"source": "server_A", "kind": "server", "latency_ms": 48, "load": 0.65, "failure_risk": 0.10},
    {"source": "mirror_B", "kind": "mirror", "latency_ms": 75, "load": 0.30, "failure_risk": 0.08},
    {"source": "peer_C", "kind": "peer", "latency_ms": 56, "load": 0.45, "failure_risk": 0.24},
])

source_csv = RESULTS / "43_source_table.csv"
sources.to_csv(source_csv, index=False)

for _, source in sources.iterrows():
    sdir = replica_root / source["source"]
    sdir.mkdir(parents=True, exist_ok=True)
    for _, chunk in chunk_plan.iterrows():
        src = REPO_ROOT / chunk["path"]
        dst = sdir / f"chunk_{int(chunk['chunk_index']):04d}.bin"
        shutil.copyfile(src, dst)

# Intentionally corrupt peer_C chunk 1.
if len(chunk_plan) > 1:
    bad_path = replica_root / "peer_C" / "chunk_0001.bin"
    bad = bytearray(bad_path.read_bytes())
    bad[20] = (bad[20] + 1) % 256
    bad_path.write_bytes(bytes(bad))

replica_rows = []
for _, chunk in chunk_plan.iterrows():
    for _, source in sources.iterrows():
        candidate_path = replica_root / source["source"] / f"chunk_{int(chunk['chunk_index']):04d}.bin"
        local_digest = sha256_file(candidate_path)
        replica_rows.append({
            "chunk_index": int(chunk["chunk_index"]),
            "chunk_identity": chunk["content_id"],
            "source": source["source"],
            "kind": source["kind"],
            "candidate_path": str(candidate_path.relative_to(REPO_ROOT)),
            "latency_ms": source["latency_ms"],
            "load": source["load"],
            "failure_risk": source["failure_risk"],
            "local_sha256": local_digest,
            "matches_identity": local_digest == chunk["sha256"],
        })

replica_table = pd.DataFrame(replica_rows)
replica_table_csv = RESULTS / "43_replica_table.csv"
replica_table.to_csv(replica_table_csv, index=False)

replica_table.head()

## 6. Discover candidates

For each chunk identity, discovery returns candidate locations.

In [ ]:
discovery_rows = []
for chunk_index, group in replica_table.groupby("chunk_index"):
    for _, row in group.iterrows():
        discovery_rows.append({
            "chunk_index": int(chunk_index),
            "content_id": row["chunk_identity"],
            "candidate": row["source"],
            "kind": row["kind"],
            "candidate_path": row["candidate_path"],
        })

discovery_table = pd.DataFrame(discovery_rows)
discovery_csv = RESULTS / "43_discovery.csv"
discovery_table.to_csv(discovery_csv, index=False)

discovery_manifest = {
    "schema": "open-model-distribution.integrated-discovery.v0",
    "artifact_content_id": artifact_id,
    "candidate_count": len(discovery_table),
    "candidates": discovery_table.to_dict(orient="records"),
}

discovery_manifest_path = RESULTS / "43_discovery_manifest.json"
discovery_manifest_path.write_text(json.dumps(discovery_manifest, indent=2), encoding="utf-8")

discovery_table.head()

## 7. Schedule transfers

Compute a transfer cost and order candidates for each chunk.

In [ ]:
alpha, beta, gamma = 1.0, 60.0, 120.0

schedule_rows = []
for _, row in replica_table.iterrows():
    cost = (
        alpha * row["latency_ms"]
        + beta * row["load"]
        + gamma * row["failure_risk"]
    )
    schedule_rows.append({
        "chunk_index": int(row["chunk_index"]),
        "source": row["source"],
        "candidate_path": row["candidate_path"],
        "expected_sha256": chunk_plan.loc[
            chunk_plan["chunk_index"] == row["chunk_index"], "sha256"
        ].iloc[0],
        "latency_ms": row["latency_ms"],
        "load": row["load"],
        "failure_risk": row["failure_risk"],
        "cost": cost,
    })

transfer_schedule = pd.DataFrame(schedule_rows).sort_values(
    ["chunk_index", "cost"]
).reset_index(drop=True)

transfer_schedule["attempt_rank"] = transfer_schedule.groupby("chunk_index").cumcount() + 1

transfer_schedule_csv = RESULTS / "43_transfer_schedule.csv"
transfer_schedule.to_csv(transfer_schedule_csv, index=False)

transfer_schedule.head(12)

## 8. Download, verify, and fallback

The scheduler suggests transfer order. Verification still decides whether a chunk is accepted. If the first candidate fails, try the next candidate for that chunk.

In [ ]:
download_records = []
accepted_records = []

for chunk_index, group in transfer_schedule.groupby("chunk_index"):
    for _, candidate in group.sort_values("attempt_rank").iterrows():
        src = REPO_ROOT / candidate["candidate_path"]
        dst = download_dir / f"chunk_{int(chunk_index):04d}_from_{candidate['source']}.bin"
        shutil.copyfile(src, dst)
        local_digest = sha256_file(dst)
        passed = local_digest == candidate["expected_sha256"]
        record = {
            "chunk_index": int(chunk_index),
            "attempt_rank": int(candidate["attempt_rank"]),
            "source": candidate["source"],
            "retrieved_path": str(dst.relative_to(REPO_ROOT)),
            "local_sha256": local_digest,
            "expected_sha256": candidate["expected_sha256"],
            "verification": "PASS" if passed else "FAIL",
            "accepted": passed,
        }
        download_records.append(record)
        if passed:
            accepted_records.append(record)
            break

download_log = pd.DataFrame(download_records)
download_log_csv = RESULTS / "43_download_log.csv"
download_log.to_csv(download_log_csv, index=False)

verification_table = pd.DataFrame(accepted_records).sort_values("chunk_index")
verification_csv = RESULTS / "43_verification.csv"
verification_table.to_csv(verification_csv, index=False)

download_log, verification_table

## 9. Reassemble and verify final artifact

Reassemble accepted chunks in manifest order, then verify the final artifact digest.

In [ ]:
reassembled_path = reassembly_dir / "reassembled-open-model.bin"

all_chunks_accepted = len(verification_table) == len(chunk_plan)

if all_chunks_accepted:
    with reassembled_path.open("wb") as handle:
        for _, chunk in chunk_plan.sort_values("chunk_index").iterrows():
            accepted = verification_table[
                verification_table["chunk_index"] == chunk["chunk_index"]
            ].iloc[0]
            handle.write((REPO_ROOT / accepted["retrieved_path"]).read_bytes())
    reassembled_digest = sha256_file(reassembled_path)
    final_pass = reassembled_digest == artifact_digest
else:
    reassembled_digest = ""
    final_pass = False

reassembly_table = pd.DataFrame([{
    "artifact": str(artifact_path.relative_to(REPO_ROOT)),
    "reassembled_path": str(reassembled_path.relative_to(REPO_ROOT)) if reassembled_path.exists() else "",
    "artifact_digest": artifact_digest,
    "reassembled_digest": reassembled_digest,
    "all_chunks_accepted": all_chunks_accepted,
    "final_verification": "PASS" if final_pass else "FAIL",
}])

reassembly_csv = RESULTS / "43_reassembly.csv"
reassembly_table.to_csv(reassembly_csv, index=False)

reassembly_table

## 10. Complete pipeline figure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.axis("off")
ax.set_title("Complete open model distribution pipeline", fontsize=17, pad=16)

nodes = [
    ("content", 0.08),
    ("address", 0.21),
    ("chunk", 0.34),
    ("replicate", 0.48),
    ("discover", 0.62),
    ("schedule", 0.76),
    ("verify +\nassemble", 0.91),
]

for label, x in nodes:
    ax.text(x, 0.58, label, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="black", lw=1.25),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(x2 - 0.055, 0.58), xytext=(x1 + 0.055, 0.58),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.22, "the final artifact is accepted only after digest verification", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
complete_pipeline_fig = FIGURES / "43_complete_pipeline.png"
fig.savefig(complete_pipeline_fig, dpi=180)
plt.show()

complete_pipeline_fig

## 11. Architecture layers figure

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.2))
ax.axis("off")
ax.set_title("Architecture layers", fontsize=17, pad=16)

layers = [
    "Application",
    "Final verification",
    "Reassembly",
    "Scheduling",
    "Discovery",
    "Replication",
    "Chunking",
    "Content addressing",
    "Storage",
]

y0 = 0.85
for idx, layer in enumerate(layers):
    y = y0 - idx * 0.085
    ax.text(0.5, y, layer, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="black", lw=1.1),
            transform=ax.transAxes)

fig.tight_layout()
architecture_layers_fig = FIGURES / "43_architecture_layers.png"
fig.savefig(architecture_layers_fig, dpi=180)
plt.show()

architecture_layers_fig

## 12. Component stack figure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis("off")
ax.set_title("Component stack", fontsize=17, pad=16)

components = [
    ("publisher", "publishes digest + manifest"),
    ("replica stores", "host chunk bytes"),
    ("discovery", "returns candidate locations"),
    ("scheduler", "orders transfer attempts"),
    ("downloader", "retrieves bytes"),
    ("verifier", "checks chunk and artifact digests"),
    ("assembler", "reconstructs the artifact"),
]

y = 0.78
for name, role in components:
    ax.text(0.30, y, name, ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
    ax.text(0.34, y, role, ha="left", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.20", fc="white", ec="black", lw=1.0),
            transform=ax.transAxes)
    y -= 0.10

fig.tight_layout()
component_stack_fig = FIGURES / "43_component_stack.png"
fig.savefig(component_stack_fig, dpi=180)
plt.show()

component_stack_fig

## 13. Chunk lifecycle figure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axis("off")
ax.set_title("Chunk lifecycle", fontsize=17, pad=16)

steps = [
    ("chunk", 0.08),
    ("identity", 0.21),
    ("replicas", 0.34),
    ("lookup", 0.47),
    ("schedule", 0.60),
    ("download", 0.74),
    ("verify", 0.87),
]

for label, x in steps:
    ax.text(x, 0.60, label, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.30", fc="white", ec="black", lw=1.15),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(steps[:-1], steps[1:]):
    ax.annotate("", xy=(x2 - 0.055, 0.60), xytext=(x1 + 0.055, 0.60),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.24, "each chunk remains tied to its digest identity throughout the workflow", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
chunk_lifecycle_fig = FIGURES / "43_chunk_lifecycle.png"
fig.savefig(chunk_lifecycle_fig, dpi=180)
plt.show()

chunk_lifecycle_fig

## 14. Parallel downloads figure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.axis("off")
ax.set_title("Parallel downloads", fontsize=17, pad=16)

for _, row in verification_table.iterrows():
    idx = int(row["chunk_index"])
    x = 0.16 + idx * 0.18
    ax.text(x, 0.70, f"chunk {idx}", ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.1),
            transform=ax.transAxes)
    ax.text(x, 0.48, row["source"], ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.1),
            transform=ax.transAxes)
    ax.annotate("", xy=(x, 0.56), xytext=(x, 0.64),
                arrowprops=dict(arrowstyle="->", lw=1.8), xycoords=ax.transAxes)

ax.text(0.5, 0.22, "chunks can transfer independently before ordered reassembly", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
parallel_downloads_fig = FIGURES / "43_parallel_downloads.png"
fig.savefig(parallel_downloads_fig, dpi=180)
plt.show()

parallel_downloads_fig

## 15. Final verification figure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.axis("off")
ax.set_title("Final verification", fontsize=17, pad=16)

nodes = [
    ("verified\nchunks", 0.18),
    ("reassemble", 0.40),
    ("SHA-256", 0.60),
    ("published\ndigest", 0.80),
]

for label, x in nodes:
    ax.text(x, 0.62, label, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.25),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(x2 - 0.07, 0.62), xytext=(x1 + 0.07, 0.62),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.28, f"final artifact verification: {'PASS' if final_pass else 'FAIL'}", ha="center", fontsize=14, transform=ax.transAxes)

fig.tight_layout()
final_verification_fig = FIGURES / "43_final_verification.png"
fig.savefig(final_verification_fig, dpi=180)
plt.show()

final_verification_fig

## 16. Final report

In [ ]:
pipeline_report = {
    "schema": "open-model-distribution.integrated-pipeline-report.v0",
    "artifact_content_id": artifact_id,
    "artifact_digest": artifact_digest,
    "chunk_count": int(len(chunk_plan)),
    "replica_source_count": int(len(sources)),
    "candidate_transfer_count": int(len(transfer_schedule)),
    "download_attempt_count": int(len(download_log)),
    "accepted_chunk_count": int(len(verification_table)),
    "all_chunks_accepted": bool(all_chunks_accepted),
    "final_verification": "PASS" if final_pass else "FAIL",
    "pipeline": [
        "content addressing",
        "verification identity",
        "chunking",
        "replication",
        "discovery",
        "scheduling",
        "download",
        "chunk verification",
        "reassembly",
        "final verification",
    ],
    "statement": "Open model distribution is accepted only after chunk verification, ordered reassembly, and final artifact digest verification.",
}

pipeline_report_path = RESULTS / "43_pipeline_report.json"
pipeline_report_path.write_text(json.dumps(pipeline_report, indent=2), encoding="utf-8")

pipeline_json_path = RESULTS / "43_pipeline.json"
pipeline_json_path.write_text(json.dumps({
    "steps": pipeline_report["pipeline"],
    "artifact_content_id": artifact_id,
    "status": pipeline_report["final_verification"],
}, indent=2), encoding="utf-8")

pipeline_report

## 17. Engineering summary

| Property | Specification | Notebook |
|---|---|---|
| Identity | SHA-256 content ID | 07 |
| Verification | Digest comparison | 13 |
| Chunking | Independent transfer units | 17 |
| Replication | Independent storage locations | 23 |
| Discovery | Candidate lookup | 29 |
| Scheduling | Ordered transfer attempts | 37 |
| Reassembly | Manifest order | 43 |
| Final verification | Artifact digest acceptance | 43 |

## 18. Engineering statement

Integrated open model distribution specifies a complete retrieval pipeline: identity states what object is requested, verification defines correctness, chunking creates transfer units, replication improves availability, discovery finds candidate locations, scheduling orders transfers, reassembly reconstructs the artifact, and final verification accepts the model for use.

## 19. Key equations

Content identity:

\[
I=H(C)
\]

Chunk identity:

\[
I_i=H(c_i)
\]

Discovery over replicas:

\[
D(I_i)\rightarrow\{L_{i1},L_{i2},\ldots\}
\]

Scheduling objective:

\[
s^\*=\arg\min_s \operatorname{cost}(s)
\]

Artifact reconstruction:

\[
C=c_0\Vert c_1\Vert\cdots\Vert c_{n-1}
\]

Final verification:

\[
H(C_{\mathrm{reassembled}})=I
\]

## 20. Notebook outputs

Notebook 43 writes:

- `results/43_integrated_pipeline/43_artifact_identity.json`
- `results/43_integrated_pipeline/43_chunk_manifest.json`
- `results/43_integrated_pipeline/43_chunk_plan.csv`
- `results/43_integrated_pipeline/43_source_table.csv`
- `results/43_integrated_pipeline/43_replica_table.csv`
- `results/43_integrated_pipeline/43_discovery.csv`
- `results/43_integrated_pipeline/43_discovery_manifest.json`
- `results/43_integrated_pipeline/43_transfer_schedule.csv`
- `results/43_integrated_pipeline/43_download_log.csv`
- `results/43_integrated_pipeline/43_verification.csv`
- `results/43_integrated_pipeline/43_reassembly.csv`
- `results/43_integrated_pipeline/43_pipeline.json`
- `results/43_integrated_pipeline/43_pipeline_report.json`
- `figures/43_complete_pipeline.png`
- `figures/43_architecture_layers.png`
- `figures/43_component_stack.png`
- `figures/43_chunk_lifecycle.png`
- `figures/43_parallel_downloads.png`
- `figures/43_final_verification.png`

In [ ]:
notebook_manifest = {
    "notebook": "43_integrated_pipeline.ipynb",
    "title": "Integrated Pipeline Specifies Open Model Distribution",
    "primary_specification": "integrated architecture",
    "statement": "Open model distribution composes identity, verification, chunking, replication, discovery, scheduling, reassembly, and final verification.",
    "equations": [
        "I=H(C)",
        "I_i=H(c_i)",
        "D(I_i)->{L_i1,L_i2,...}",
        "s*=argmin cost(s)",
        "C=c_0||...||c_{n-1}",
        "H(C_reassembled)=I",
    ],
    "outputs": {
        "artifact_identity": str(artifact_identity_path.relative_to(REPO_ROOT)),
        "chunk_manifest": str(chunk_manifest_path.relative_to(REPO_ROOT)),
        "chunk_plan_csv": str(chunk_plan_csv.relative_to(REPO_ROOT)),
        "source_table_csv": str(source_csv.relative_to(REPO_ROOT)),
        "replica_table_csv": str(replica_table_csv.relative_to(REPO_ROOT)),
        "discovery_csv": str(discovery_csv.relative_to(REPO_ROOT)),
        "discovery_manifest": str(discovery_manifest_path.relative_to(REPO_ROOT)),
        "transfer_schedule_csv": str(transfer_schedule_csv.relative_to(REPO_ROOT)),
        "download_log_csv": str(download_log_csv.relative_to(REPO_ROOT)),
        "verification_csv": str(verification_csv.relative_to(REPO_ROOT)),
        "reassembly_csv": str(reassembly_csv.relative_to(REPO_ROOT)),
        "pipeline_json": str(pipeline_json_path.relative_to(REPO_ROOT)),
        "pipeline_report": str(pipeline_report_path.relative_to(REPO_ROOT)),
        "complete_pipeline_figure": str(complete_pipeline_fig.relative_to(REPO_ROOT)),
        "architecture_layers_figure": str(architecture_layers_fig.relative_to(REPO_ROOT)),
        "component_stack_figure": str(component_stack_fig.relative_to(REPO_ROOT)),
        "chunk_lifecycle_figure": str(chunk_lifecycle_fig.relative_to(REPO_ROOT)),
        "parallel_downloads_figure": str(parallel_downloads_fig.relative_to(REPO_ROOT)),
        "final_verification_figure": str(final_verification_fig.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "43_notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

notebook_manifest

## 21. Download artifacts

Run the final cell to package notebook 43 outputs for download.

In [ ]:
# Package notebook artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

NOTEBOOKS = REPO_ROOT / "notebooks"
SITE = REPO_ROOT / "site"

zip_path = RESULTS / "43_integrated_pipeline_artifacts.zip"

notebook_path = NOTEBOOKS / "43_integrated_pipeline.ipynb"
fallback_notebook_path = Path.cwd() / "43_integrated_pipeline.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)

        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)

        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass